# Исследование данных по гипотезам

In [ ]:
'''
Что уже известно:
- 222 признака дают 0.722 на кросс-валидации и 0.718 на Public; признаки вышли на плато;
- работает время заказов и группа «товары»; деньги, скидки и лояльность почти ничего не несут;
- у клиентов с 2–3 днями заказов target почти предопределён днём «последний заказ + средний интервал»
  (зона 25..92 дня после отсечения); для 4+ дней заказов правила нет;
- отвергнуто: утечки в ID, сдвинутые срезы, ряды покупок по категориям, визиты, сырые даты,
  последние заказы «как есть», счётчики подкатегорий, траектория лояльности, признаки брендов.

Как проверяются гипотезы в этом ноутбуке:
- описательно: таблица долей target = 1 с числом клиентов в ячейках;
- предсказательно: блок признаков добавляется к 222 существующим, LightGBM на одних и тех же трёх фолдах;
  блок принимается, если плюс есть минимум в 8 парах «фолд x срез» из 9 и средний прирост больше 0.002.
Все предсказательные проверки попадают в итоговую таблицу в последнем разделе.

Разделы:
1. как устроены и связаны между собой колонки; 2. что такое «заказ»; 3. двойники среди клиентов;
4. нетипичные клиенты; 5. срезы и тест: что меняется во времени; 6. правило для 2–3 заказов с точностью до дня;
7. уверенные ошибки модели; 8. повторяемость покупок; 9. итоговая таблица.
'''
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 3
ZONE_LO, ZONE_HI = 25, 92
MIN_GAIN, MIN_WINS = 0.002, 8
N_EDGES = [0, 1, 2, 3, 5, 9, 20, 10_000]
N_LABELS = ["1", "2", "3", "4-5", "6-9", "10-20", "21+"]
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)

data = data.drop(columns=["top_brand"])
data["top_category2"] = data["top_category2"].astype("category")
ALL = [c for c in feats.columns if c != "top_brand"]
cid = "customer_id"

tr = data[data["part"].eq("train")].reset_index(drop=True)
te = data[data["part"].eq("test")].reset_index(drop=True)
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()
slices = np.unique(sl)
tr["n_bin"] = pd.cut(tr["n_days"], N_EDGES, labels=N_LABELS)
tgt = tr.set_index(cid)["target"]

# --- единая быстрая проверка блоков признаков
PARAMS = dict(objective="binary", learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=100, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50, random_state=SEED, n_jobs=-1, verbose=-1)
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(tr, tr["slice"] + "_" + tr["target"].astype(int).astype(str)))
RUNS, VERDICTS = {}, []


def run(name, features, frame=None):
    frame = tr if frame is None else frame
    p, imp = np.zeros(len(frame)), pd.Series(0.0, index=features)
    for a, b in folds:
        m = lgb.LGBMClassifier(**PARAMS).fit(frame[features].iloc[a], y[a], eval_set=[(frame[features].iloc[b], y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        p[b] = m.predict_proba(frame[features].iloc[b])[:, 1]
        g = pd.Series(m.booster_.feature_importance("gain"), index=features)
        imp += g / g.sum() / len(folds)
    cells = np.array([[fast_ap(y[b][sl[b] == s], p[b][sl[b] == s]) for s in slices] for _, b in folds])
    RUNS[name] = dict(ap=float(np.mean([fast_ap(y[sl == s], p[sl == s]) for s in slices])), cells=cells, imp=imp, oof=p)
    return RUNS[name]


def test_block(section, name, block, base="все 222"):
    # block: DataFrame с индексом customer_id; проверяем прирост от добавления его колонок к базовому набору
    block = block.astype("float32")
    frame = tr.join(block.reindex(tr[cid]).set_axis(tr.index), rsuffix="__new")
    new_cols = [c if c not in tr.columns else c + "__new" for c in block.columns]
    r = run(f"{base} + {name}", ALL + new_cols, frame)
    d = (r["cells"] - RUNS[base]["cells"]).ravel()
    ok = d.mean() > MIN_GAIN and (d > 0).sum() >= MIN_WINS
    VERDICTS.append({"раздел": section, "гипотеза": name, "признаков": len(new_cols), "AP@20%": r["ap"], "прирост": d.mean(), "разброс": d.std(),
                     "плюс в парах": f"{int((d > 0).sum())} из {len(d)}", "вердикт": "принята" if ok else "отклонена"})
    top = r["imp"][new_cols].sort_values(ascending=False).head(6)
    print(f"[{name}] прирост {d.mean():+.4f} ± {d.std():.4f}, плюс в {int((d > 0).sum())} из {len(d)} -> {'ПРИНЯТА' if ok else 'отклонена'}; "
          f"самые важные новые: {({k: round(float(v), 4) for k, v in top.items()})}")


ref = run("все 222", ALL)
tr["p_ref"] = ref["oof"]
tr["pct_ref"] = tr.groupby("slice")["p_ref"].rank(pct=True, ascending=False)
print(f"Опорная модель, 222 признака: AP@20% {ref['ap']:.4f} | подготовка {(time.time() - T0) / 60:.1f} мин")

# --- таблица заказов: одна строка на заказ, самые свежие первыми
lines["full_price"] = ((lines["discount_percent"] == 0) & (lines["loyalty_percent"] == 0)).astype("float32")
main = lines.sort_values([cid, "order_id", "paid_price"], ascending=[True, True, False]).drop_duplicates([cid, "order_id"]).set_index("order_id")
o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), date=("order_date", "first"), order_sum=("paid_price", "sum"), n_items=("paid_price", "size"), price_max=("paid_price", "max"),
         disc=("discount_percent", "mean"), loy=("loyalty_percent", "max"), loy_min=("loyalty_percent", "min"), n_cat2=("category2", "nunique"))
    .reset_index()
)
o["main_cat"] = o["order_id"].map(main["category"].astype(str))
o = o.sort_values([cid, "d", "order_id"]).reset_index(drop=True)
o["rk"] = o.groupby(cid).cumcount()
o["gap_prev"] = o.groupby(cid)["d"].shift(-1) - o["d"]      # дней с предыдущего по времени заказа
o["prior_n"] = o.groupby(cid)["rk"].transform("max") - o["rk"]   # сколько заказов было раньше в видимой истории
o["prior_sum"] = o.iloc[::-1].groupby(cid)["order_sum"].cumsum().iloc[::-1] - o["order_sum"]

## Как устроены и связаны колонки

In [ ]:
smp = lines.sample(min(len(lines), 300_000), random_state=SEED)
print("Позиции заказов: корреляция Спирмена")
display(smp.assign(date_num=(smp["order_date"] - smp["order_date"].min()).dt.days)[["paid_price", "discount_percent", "loyalty_percent", "date_num", "d"]].corr(method="spearman"))
print("Цена, скидка и лояльность по категориям и полу товара:")
for col in ["category", "gender_product"]:
    display(lines.groupby(col).agg(позиций=("paid_price", "size"), цена_медиана=("paid_price", "median"), скидка_средняя=("discount_percent", "mean"),
                                   без_скидки=("discount_percent", lambda s: (s == 0).mean()), лояльность_средняя=("loyalty_percent", "mean"),
                                   без_лояльности=("loyalty_percent", lambda s: (s == 0).mean())))

print("От чего зависит лояльность заказа: корреляция Спирмена на уровне заказа")
display(o.assign(date_num=(o["date"] - o["date"].min()).dt.days)[["loy", "prior_n", "prior_sum", "order_sum", "n_items", "disc", "date_num"]].corr(method="spearman").loc[["loy", "disc"]])
print("Средняя лояльность заказа по числу предыдущих заказов и по кварталу:")
display(o.groupby(pd.cut(o["prior_n"], [-1, 0, 1, 2, 4, 9, 19, 1000]), observed=True)["loy"].agg(["mean", lambda s: (s > 0).mean(), "size"]).set_axis(["средняя", "доля > 0", "заказов"], axis=1).T)
display(o.groupby(o["date"].dt.to_period("Q"))["loy"].agg(["mean", lambda s: (s > 0).mean(), "size"]).set_axis(["средняя", "доля > 0", "заказов"], axis=1).T)
print("Значения лояльности на уровне заказа (доля заказов):", o["loy"].value_counts(normalize=True).head(12).round(4).to_dict())
print("Доля заказов, где у позиций разная лояльность:", round(float((o["loy"] != o["loy_min"]).mean()), 4))

key = ["recency", "tenure", "n_days", "gap_mean", "tnext_mean", "sum_paid", "order_sum_mean", "items_per_order", "disc_mean", "loy_mean", "n_brands", "n_cat2",
       "ser_category2_n", "ser_brand_n", "repeat_product_share", "multi_order_days"]
print("Клиенты train: корреляция Спирмена ключевых признаков с target и между собой")
display(tr[["target"] + key].corr(method="spearman").round(2))

## Что такое «заказ»

In [ ]:
print("Интервал между соседними заказами клиента, дни (доля пар заказов):")
gp = o["gap_prev"].dropna()
display(pd.cut(gp, [-1, 0, 1, 3, 7, 14, 30, 60, 90, 180, 800]).value_counts(normalize=True).sort_index().to_frame("доля").T)

# пары «заказ и предыдущий», идущие с разрывом до 3 дней: похожи ли они по содержимому
o["prev_order"] = o.groupby(cid)["order_id"].shift(-1)
close = o[o["gap_prev"].le(3)][[cid, "order_id", "prev_order", "gap_prev", "order_sum", "n_items"]].copy()
far = o[o["gap_prev"].gt(30)][[cid, "order_id", "prev_order", "gap_prev", "order_sum", "n_items"]].copy()
lp = lines[["order_id", "product_id", "category2", "brand"]]


def overlap(pairs, col):
    a = lp[["order_id", col]].drop_duplicates()
    m = pairs[["order_id", "prev_order"]].merge(a, on="order_id").merge(a.rename(columns={"order_id": "prev_order"}), on=["prev_order", col])
    return pairs["order_id"].isin(m["order_id"]).mean()


print("Доля пар соседних заказов с общим товаром / подкатегорией / брендом:")
display(pd.DataFrame({name: {c: overlap(p, c) for c in ["product_id", "category2", "brand"]} | {"пар": len(p)} for name, p in [("разрыв до 3 дней", close), ("разрыв больше 30 дней", far)]}).T)

quick = o.assign(quick=o["gap_prev"].le(3).astype(float), same_day=o["gap_prev"].eq(0).astype(float))
q = quick.groupby(cid).agg(quick_share=("quick", "mean"), n_quick=("quick", "sum"), n_same_day=("same_day", "sum"), n_orders=("order_id", "size"))
q["last_is_quick"] = quick[quick["rk"] == 0].set_index(cid)["quick"]
same_prod = close[close["order_id"].isin(close.merge(lp[["order_id", "product_id"]], on="order_id").merge(
    lp[["order_id", "product_id"]].rename(columns={"order_id": "prev_order"}), on=["prev_order", "product_id"])["order_id"])]
q["n_same_product_reorders"] = same_prod.groupby(cid).size()
q["n_same_product_reorders"] = q["n_same_product_reorders"].fillna(0)
q["tiny_orders"] = (o["order_sum"] < o["order_sum"].quantile(0.05)).groupby(o[cid]).sum()
q["orders_per_day"] = q["n_orders"] / tr.set_index(cid)["n_days"].reindex(q.index).fillna(te.set_index(cid)["n_days"].reindex(q.index))
print("Доля target = 1 по доле «быстрых» повторных заказов (до 3 дней после предыдущего), по числу дней с заказами:")
display(tr.assign(qb=pd.cut(tr[cid].map(q["quick_share"]), [-0.01, 0, 0.1, 0.25, 0.5, 1.0])).pivot_table(index="qb", columns="n_bin", values="target", aggfunc="mean", observed=True))
test_block("заказ", "быстрые повторные заказы", q.drop(columns=["n_orders"]))

## Двойники среди клиентов

In [ ]:
# Один и тот же человек под разными customer_id покупал бы те же товары в те же даты по той же цене.
tk = ["product_id", "order_date", "paid_price"]
trip = lines[[cid] + tk].drop_duplicates()
n_c = trip.groupby(tk)[cid].transform("nunique")
shared = trip[n_c.between(2, 5)]
print(f"Троек (товар, дата, цена) всего: {len(trip):,}; встречаются у 2–5 разных клиентов: {shared.groupby(tk).ngroups:,}; у 6+ клиентов: {trip[n_c > 5].groupby(tk).ngroups:,}")
pairs = shared.merge(shared, on=tk)
pairs = pairs[pairs[f"{cid}_x"] < pairs[f"{cid}_y"]]
pc = pairs.groupby([f"{cid}_x", f"{cid}_y"]).agg(общих_позиций=("product_id", "size"), общих_дат=("order_date", "nunique")).reset_index()
print("Пары клиентов по числу общих позиций:", pc["общих_позиций"].clip(upper=6).value_counts().sort_index().to_dict() if len(pc) else "пар нет")
if len(pc):
    info = cust.set_index(cid)
    n_lines = lines.groupby(cid).size()
    for side in ["x", "y"]:
        pc[f"срез_{side}"] = pc[f"{cid}_{side}"].map(info["slice"])
        pc[f"target_{side}"] = pc[f"{cid}_{side}"].map(info["target"])
        pc[f"позиций_{side}"] = pc[f"{cid}_{side}"].map(n_lines)
    pc["доля_общего"] = pc["общих_позиций"] / pc[["позиций_x", "позиций_y"]].min(axis=1)
    strong = pc[(pc["общих_позиций"] >= 3) & (pc["общих_дат"] >= 2)]
    print(f"Пар с 3+ общими позициями в 2+ разные даты: {len(strong)}; из них между разными срезами: {(strong['срез_x'] != strong['срез_y']).sum()}")
    print("Пары с наибольшим пересечением:")
    display(pc.sort_values(["общих_позиций", "доля_общего"], ascending=False).head(15))
    both = strong.dropna(subset=["target_x", "target_y"])
    if len(both):
        print(f"У пар из train target совпадает в {(both['target_x'] == both['target_y']).mean():.1%} случаев ({len(both)} пар)")
    twins = pd.concat([strong[f"{cid}_x"], strong[f"{cid}_y"]]).unique()
    print("Клиентов, у которых есть двойник:", len(twins), "| из них в test:", int(pd.Series(twins).map(info["part"]).eq("test").sum()))

## Нетипичные клиенты

In [ ]:
gcust = o.groupby(cid)
odd = pd.DataFrame({
    "заказ на 15+ позиций": gcust["n_items"].max() >= 15,
    "40+ заказов": gcust.size() >= 40,
    "5+ заказов в один день": o.groupby([cid, "d"]).size().groupby(cid).max() >= 5,
    "в среднем 6+ позиций в заказе": gcust["n_items"].mean() >= 6,
    "сумма покупок в верхнем 1%": gcust["order_sum"].sum() >= gcust["order_sum"].sum().quantile(0.99),
    "один товар куплен 4+ раза": lines.groupby([cid, "product_id"]).size().groupby(cid).max() >= 4,
}).fillna(False)
odd["любой признак"] = odd.any(axis=1)
flags = odd.reindex(tr[cid]).set_axis(tr.index)
rows = {}
for col in flags.columns:
    m = flags[col].to_numpy()
    rows[col] = {"клиентов": int(m.sum()), "доля target = 1": y[m].mean() if m.any() else np.nan, "средний прогноз модели": tr.loc[m, "p_ref"].mean() if m.any() else np.nan,
                 "доля в топ-20%": (tr.loc[m, "pct_ref"] <= 0.2).mean() if m.any() else np.nan, "target в топ-20%": y[m & (tr["pct_ref"] <= 0.2).to_numpy()].mean() if m.any() else np.nan}
rows["все клиенты"] = {"клиентов": len(tr), "доля target = 1": y.mean(), "средний прогноз модели": tr["p_ref"].mean(), "доля в топ-20%": 0.2, "target в топ-20%": y[(tr["pct_ref"] <= 0.2).to_numpy()].mean()}
display(pd.DataFrame(rows).T)

## Срезы и тест: что меняется во времени

In [ ]:
def adversarial(a, b, features):
    X = pd.concat([a[features], b[features]], ignore_index=True)
    lab = np.r_[np.zeros(len(a)), np.ones(len(b))]
    p, imp = np.zeros(len(X)), pd.Series(0.0, index=features)
    for i, j in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, lab):
        m = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300}).fit(X.iloc[i], lab[i])
        p[j] = m.predict_proba(X.iloc[j])[:, 1]
        g = pd.Series(m.booster_.feature_importance("gain"), index=features)
        imp += g / g.sum() / 3
    return roc_auc_score(lab, p), imp.sort_values(ascending=False).head(10)


print("Можно ли по признакам отличить один срез от другого (AUC 0.5 — нельзя):")
for name, a, b in [(f"{slices[0]} против {slices[-1]}", tr[sl == slices[0]], tr[sl == slices[-1]]), (f"{slices[-1]} против теста", tr[sl == slices[-1]], te), ("весь train против теста", tr, te)]:
    auc, top = adversarial(a, b, ALL)
    print(f"  {name}: AUC {auc:.4f}; главные отличия: {({k: round(float(v), 3) for k, v in top.items()})}")

print("Перенос между срезами: обучение на строке, AP@20% на столбце")
mat = pd.DataFrame(index=slices, columns=slices, dtype=float)
for s_tr in slices:
    a = np.flatnonzero(sl == s_tr)
    i_fit, i_es = a[np.arange(len(a)) % 7 != 0], a[np.arange(len(a)) % 7 == 0]   # каждая седьмая строка — для ранней остановки и для диагонали
    m = lgb.LGBMClassifier(**PARAMS).fit(tr[ALL].iloc[i_fit], y[i_fit], eval_set=[(tr[ALL].iloc[i_es], y[i_es])], callbacks=[lgb.early_stopping(100, verbose=False)])
    for s_te in slices:
        b = i_es if s_te == s_tr else np.flatnonzero(sl == s_te)
        mat.loc[s_tr, s_te] = fast_ap(y[b], m.predict_proba(tr[ALL].iloc[b])[:, 1])
display(mat)
print("Доля target = 1 в ключевых ячейках по срезам (устойчива ли зависимость во времени):")
cells = tr.assign(rec=pd.cut(tr["recency"], [-1, 14, 60, 180, 365]))
display(cells.pivot_table(index=["n_bin", "rec"], columns="slice", values="target", aggfunc="mean", observed=True))

## Правило для 2–3 заказов с точностью до дня

In [ ]:
two = tr[tr["n_days"].eq(2)]
three = tr[tr["n_days"].eq(3)]
prof = pd.DataFrame({"2 дня: доля": two.groupby(pd.cut(two["tnext_mean"], range(0, 121, 3)), observed=False)["target"].mean(),
                     "2 дня: клиентов": two.groupby(pd.cut(two["tnext_mean"], range(0, 121, 3)), observed=False)["target"].size(),
                     "3 дня: доля": three.groupby(pd.cut(three["tnext_mean"], range(0, 121, 3)), observed=False)["target"].mean(),
                     "3 дня: клиентов": three.groupby(pd.cut(three["tnext_mean"], range(0, 121, 3)), observed=False)["target"].size()})
print("Доля target = 1 по ожидаемому дню следующего заказа, шаг 3 дня:")
display(prof.T)
print("Два дня с заказами: то же по срезам, шаг 10 дней (длина окна у срезов 90, 89 и 92 дня):")
display(two.pivot_table(index=pd.cut(two["tnext_mean"], range(0, 131, 10)), columns="slice", values="target", aggfunc="mean", observed=False).T)
print("Два дня с заказами: длина интервала (строки) x ожидаемый день (столбцы)")
display(two.pivot_table(index=pd.cut(two["gap_mean"], [0, 30, 45, 60, 80, 100, 130, 180, 800]), columns=pd.cut(two["tnext_mean"], [-800, 0, 15, 25, 35, 50, 65, 80, 92, 105, 800]),
                        values="target", aggfunc="mean", observed=False))
irr = (three["gap_max"] - three["gap_min"]) / (three["gap_max"] + three["gap_min"])
inz = three["tnext_mean"].gt(ZONE_LO) & three["tnext_mean"].le(ZONE_HI)
print("Три дня с заказами: доля target = 1 в зоне и вне её по неравномерности двух интервалов (0 — равные):")
display(three.assign(irr=pd.cut(irr, [-0.01, 0.1, 0.25, 0.5, 0.75, 1.0]), zone=np.where(inz, "в зоне", "вне зоны")).pivot_table(index="irr", columns="zone", values="target", aggfunc=["mean", "size"], observed=True))

full = pd.concat([tr, te], ignore_index=True)
small = full["n_days"].le(3)
t = full["tnext_mean"].where(small)
zone = pd.DataFrame({"z_in": (t.gt(ZONE_LO) & t.le(ZONE_HI)).astype(float).where(t.notna()), "z_from_lo": t - ZONE_LO, "z_to_hi": ZONE_HI - t,
                     "z_dist": (ZONE_LO - t).clip(lower=0) + (t - ZONE_HI).clip(lower=0), "z_center": (t - (ZONE_LO + ZONE_HI) / 2).abs(),
                     "z_gap": full["gap_mean"].where(small), "z_irr": ((full["gap_max"] - full["gap_min"]) / (full["gap_max"] + full["gap_min"])).where(small)})
zone.index = full[cid]
test_block("правило", "точные границы зоны для 1–3 дней заказов", zone)

## Уверенные ошибки модели

In [ ]:
num = [c for c in ALL if c != "top_category2"]


def contrast(mask_a, mask_b, label_a, label_b, k=20):
    # какие признаки сильнее всего различают две группы клиентов (AUC одного признака, 0.5 — не различает)
    out = {}
    lab = np.r_[np.ones(mask_a.sum()), np.zeros(mask_b.sum())]
    for c in num:
        v = np.r_[tr.loc[mask_a, c].to_numpy(), tr.loc[mask_b, c].to_numpy()]
        ok = ~np.isnan(v)
        if ok.sum() > 100 and len(np.unique(lab[ok])) == 2 and np.nanstd(v) > 0:
            out[c] = {"AUC": roc_auc_score(lab[ok], v[ok]), f"медиана: {label_a}": np.nanmedian(tr.loc[mask_a, c]), f"медиана: {label_b}": np.nanmedian(tr.loc[mask_b, c])}
    d = pd.DataFrame(out).T
    return d.reindex((d["AUC"] - 0.5).abs().sort_values(ascending=False).index).head(k)


top = (tr["pct_ref"] <= 0.10).to_numpy()
band = ((tr["pct_ref"] > 0.10) & (tr["pct_ref"] <= 0.30)).to_numpy()
print(f"Верхние 10% рейтинга: {top.sum()} клиентов, купили {y[top].mean():.3f}. Чем не купившие отличаются от купивших:")
display(contrast(top & (y == 0), top & (y == 1), "не купил", "купил"))
print(f"Полоса 10–30% рейтинга: {band.sum()} клиентов, купили {y[band].mean():.3f}. Чем купившие отличаются от не купивших:")
display(contrast(band & (y == 1), band & (y == 0), "купил", "не купил"))

# Остался ли сигнал внутри верхних 30%: вторая модель только на этих клиентах, с прогнозом первой как признаком
sub = np.flatnonzero(tr["pct_ref"] <= 0.30)
Xs, ys = tr.iloc[sub][ALL + ["p_ref"]], y[sub]
p2 = np.zeros(len(sub))
for i, j in StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xs, ys):
    m = lgb.LGBMClassifier(**{**PARAMS, "num_leaves": 15}).fit(Xs.iloc[i], ys[i], eval_set=[(Xs.iloc[j], ys[j])], callbacks=[lgb.early_stopping(100, verbose=False)])
    p2[j] = m.predict_proba(Xs.iloc[j])[:, 1]
print(f"Внутри верхних 30%: AUC прогноза первой модели {roc_auc_score(ys, tr['p_ref'].to_numpy()[sub]):.4f}, AUC второй модели {roc_auc_score(ys, p2):.4f} "
      "(если они равны, признаки внутри топа исчерпаны)")
print("AUC первой модели внутри групп по числу дней с заказами:")
display(pd.Series({nb: roc_auc_score(y[(tr["n_bin"] == nb).to_numpy()], tr.loc[tr["n_bin"] == nb, "p_ref"]) for nb in N_LABELS if (tr["n_bin"] == nb).sum() > 50}).to_frame("AUC").T)


def show(ids, title):
    print(f"\n--- {title}. Формат заказа: дней до отсечения / сумма / позиций / категория")
    for c in ids:
        h = o[o[cid] == c].head(14)
        print(f"[{tr.loc[tr[cid] == c, 'pct_ref'].iloc[0]:.3f}] " + "  ".join(f"{int(a)}/{b:.0f}/{int(n)}/{str(k)[:4]}" for a, b, n, k in zip(h["d"], h["order_sum"], h["n_items"], h["main_cat"])))


rng = np.random.default_rng(SEED)
fp = tr[(tr["pct_ref"] <= 0.05) & (tr["target"] == 0)][cid].to_numpy()
tp = tr[(tr["pct_ref"] <= 0.05) & (tr["target"] == 1)][cid].to_numpy()
fn = tr[(tr["pct_ref"] > 0.50) & (tr["target"] == 1) & (tr["n_days"] >= 4)][cid].to_numpy()
show(rng.choice(fp, min(15, len(fp)), replace=False), "Верхние 5% рейтинга, НЕ купили")
show(rng.choice(tp, min(10, len(tp)), replace=False), "Верхние 5% рейтинга, купили")
show(rng.choice(fn, min(15, len(fn)), replace=False), "Нижняя половина рейтинга, 4+ дней с заказами, КУПИЛИ")

## Повторяемость покупок

In [ ]:
# В группе «товары» два вида признаков: состав покупок и ряды повторных покупок (ser_*). Что из них работает?
comp = [c for c in ALL if re.search(r"^(cat_share|gender_share|top_|n_products|n_brands|n_cat|n_cat2|n_gender|brands_per_item|cat2_per_item)", c)]
ser = [c for c in ALL if c.startswith("ser_")]
for name, drop in [("все 222 без состава покупок", comp), ("все 222 без рядов повторных покупок", ser)]:
    r = run(name, [c for c in ALL if c not in drop])
    d = (r["cells"] - RUNS["все 222"]["cells"]).ravel()
    print(f"{name} ({len(drop)} признаков убрано): {d.mean():+.4f} ± {d.std():.4f}, плюс в {int((d > 0).sum())} из {len(d)}")

rep = pd.DataFrame(index=cust[cid])
for col in ["category2", "brand", "product_id", "category"]:
    st = lines[[cid, col, "d"]].drop_duplicates().groupby([cid, col], observed=True)["d"].agg(n="size", last="min", first="max").reset_index()
    g = st.groupby(cid)
    for k in [2, 3, 5]:
        rep[f"rep_{col}_ge{k}"] = (st["n"] >= k).groupby(st[cid]).sum()
    rep[f"rep_{col}_max_days"] = g["n"].max()
    rep[f"rep_{col}_share"] = (st["n"] >= 2).groupby(st[cid]).mean()
    multi = st[st["n"] >= 2]
    rep[f"rep_{col}_last_d"] = multi.groupby(cid)["last"].min()                 # когда в последний раз покупал то, что уже покупал раньше
    rep[f"rep_{col}_span_mean"] = (multi["first"] - multi["last"]).groupby(multi[cid]).mean()
    rep[f"rep_{col}_recent90"] = (multi["last"] <= 90).groupby(multi[cid]).sum()
test_block("повторяемость", "расширенные признаки повторяемости", rep)

## Итоговая таблица

In [ ]:
print(f"Опорная модель: {RUNS['все 222']['ap']:.4f}. Критерий: прирост больше {MIN_GAIN} и плюс минимум в {MIN_WINS} парах из 9.")
display(pd.DataFrame(VERDICTS))
print(f"Всего времени: {(time.time() - T0) / 60:.1f} мин")